# 02 短期负荷预测建模

题目四：城市电力负荷分析与预测

本 notebook 负责：读入清洗后的数据，构造时间、气象、滞后特征，训练并比较**多元线性回归、随机森林、梯度提升**三种模型，用 RMSE / MAE / R² 评估，输出报告图 9–12（`fig_B1` ~ `fig_B4`）。

环境：Python 3.9.12 / pandas 2.3.3 / scikit-learn 1.6.1 / matplotlib 3.9.4（我的电脑本机）。

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Windows 下显示中文与负号
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print('pandas', pd.__version__, '| sklearn', __import__('sklearn').__version__)

pandas 3.0.3 | sklearn 1.9.0


## 1. 读入数据与目标设定

读入与清洗口径（`01_eda.ipynb`）：重命名列为短名、显式解析时间、设为索引并升序排列。

预测目标设为**三区总负荷 `Z_total = Z1 + Z2 + Z3`**（题目背景为电网调度提供参考，调度关心全市总量）；预测步长为 **t+1（10 分钟后，one-step-ahead）**。

In [2]:
df = pd.read_csv('data/题4_城市电力负荷.csv', encoding='utf-8-sig')
df.columns = ['DateTime', 'T', 'RH', 'WS', 'GDF', 'DF', 'Z1', 'Z2', 'Z3']
df['DateTime'] = pd.to_datetime(df['DateTime'], format='%m/%d/%Y %H:%M')
df = df.set_index('DateTime').sort_index()

# 目标：三区总负荷
df['Z_total'] = df[['Z1', 'Z2', 'Z3']].sum(axis=1)
print('shape:', df.shape)
print('时间范围:', df.index.min(), '->', df.index.max())
print('总负荷均值: %.1f kW' % df['Z_total'].mean())

shape: (52416, 9)
时间范围: 2017-01-01 00:00:00 -> 2017-12-30 23:50:00
总负荷均值: 71222.9 kW


## 2. 特征工程

构造三类时间特征（小时、星期、月份），其中周期量用 sin/cos 编码，使线性模型也能理解“23 时与 0 时相邻”这类周期关系；并加入气象特征与**滞后特征（自回归项）**——10 分钟、1 小时、1 天、1 周前的总负荷。

In [3]:
# 2.1 时间特征（含周期编码）
df['hour'] = df.index.hour
df['dow'] = df.index.dayofweek
df['month'] = df.index.month
df['is_weekend'] = (df['dow'] >= 5).astype(int)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)
df['dow_sin'] = np.sin(2 * np.pi * df['dow'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['dow'] / 7)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

# 2.2 滞后特征（自回归项）：10min / 1h / 1天 / 1周 前
LAGS = [1, 6, 144, 1008]
for lag in LAGS:
    df[f'lag_{lag}'] = df['Z_total'].shift(lag)

# 两组特征：用于“有无滞后”这一维度的对比
WEATHER = ['T', 'RH', 'WS', 'GDF', 'DF']
TIME = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend']
LAG_COLS = [f'lag_{l}' for l in LAGS]
FEAT_NO_LAG = WEATHER + TIME           # 仅气象 + 时间
FEAT_WITH_LAG = WEATHER + TIME + LAG_COLS  # 再加滞后
print('特征数：无滞后 %d 个 / 含滞后 %d 个' % (len(FEAT_NO_LAG), len(FEAT_WITH_LAG)))

特征数：无滞后 12 个 / 含滞后 16 个


## 3. 时间顺序划分

时间序列必须**按时间顺序**划分，不能随机打乱（否则会“用未来预测过去”造成数据泄漏）。这里取前 10 个月（1–10 月）训练、后 2 个月（11–12 月）测试；滞后特征会让训练集开头 1008 行出现 NaN，一并剔除。

In [4]:
train = df[df.index < '2017-11-01']
test = df[df.index >= '2017-11-01']
print('训练集:', train.index.min().date(), '->', train.index.max().date(), len(train), '行')
print('测试集:', test.index.min().date(), '->', test.index.max().date(), len(test), '行')

# 剔除滞后特征产生的 NaN（仅影响训练集开头 1008 行）
train = train.dropna(subset=LAG_COLS)
test = test.dropna(subset=LAG_COLS)
print('dropna 后 训练:', len(train), '| 测试:', len(test))

y_train = train['Z_total'].values
y_test = test['Z_total'].values

训练集: 2017-01-01 -> 2017-10-31 43776 行
测试集: 2017-11-01 -> 2017-12-30 8640 行
dropna 后 训练: 42768 | 测试: 8640


## 4. 模型训练与评估

三个模型：**多元线性回归**（基准下限，训练前标准化）、**随机森林**（非线性集成）、**梯度提升**（`HistGradientBoostingRegressor`，比 LightGBM 免额外装包、效果同档）。每个模型都在“仅气象+时间”与“气象+时间+滞后”两组特征上各训一次，得到两条对比维度。

In [5]:
def fit_predict(model, Xtr, ytr, Xte, scale=False):
    if scale:  # 线性回归需标准化，树模型不需要
        sc = StandardScaler()
        Xtr = sc.fit_transform(Xtr)
        Xte = sc.transform(Xte)
    model.fit(Xtr, ytr)
    return model.predict(Xte)

def metrics(y_true, y_pred):
    return {
        'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'MAE': float(mean_absolute_error(y_true, y_pred)),
        'R2': float(r2_score(y_true, y_pred)),
    }

models = {
    '线性回归': (LinearRegression(), True),
    '随机森林': (RandomForestRegressor(n_estimators=300, max_depth=20, n_jobs=-1, random_state=42), False),
    '梯度提升': (HistGradientBoostingRegressor(random_state=42), False),
}

rows, pred_store = [], {}
for mname, (model, need_scale) in models.items():
    for tag, feats in [('仅气象+时间', FEAT_NO_LAG), ('气象+时间+滞后', FEAT_WITH_LAG)]:
        pred = fit_predict(model, train[feats], y_train, test[feats], scale=need_scale)
        m = metrics(y_test, pred)
        m.update({'模型': mname, '特征': tag})
        rows.append(m)
        pred_store[(mname, tag)] = pred

res = pd.DataFrame(rows)[['模型', '特征', 'RMSE', 'MAE', 'R2']]
res['相对RMSE'] = (res['RMSE'] / df['Z_total'].mean() * 100).round(2)
# 整齐对齐的表格输出（中文按双宽度对齐，数值列右对齐）
def _w(s):
    return sum(2 if ord(ch) > 127 else 1 for ch in str(s))

def _pad(s, w, right=False):
    s = str(s)
    gap = w - _w(s)
    return (' ' * gap + s) if right else (s + ' ' * gap)

def _print_aligned(df, right=('RMSE', 'MAE', 'R²', '相对RMSE')):
    cols = list(df.columns)
    width = [max(_w(c), max(_w(v) for v in df[c])) for c in cols]
    lines = ['  '.join(_pad(c, width[j], c in right) for j, c in enumerate(cols))]
    for _, r in df.iterrows():
        lines.append('  '.join(_pad(r[c], width[j], c in right) for j, c in enumerate(cols)))
    print('\n'.join(lines))

disp = res.copy().rename(columns={'R2': 'R²'})
disp['RMSE'] = disp['RMSE'].map(lambda v: f'{v:,.0f}')
disp['MAE'] = disp['MAE'].map(lambda v: f'{v:,.0f}')
disp['R²'] = disp['R²'].map(lambda v: f'{v:.4f}')
disp['相对RMSE'] = disp['相对RMSE'].map(lambda v: f'{v:.2f}%')
_print_aligned(disp, right=('RMSE', 'MAE', 'R²', '相对RMSE'))

  模型       特征    RMSE     MAE   R2  相对RMSE
线性回归   仅气象+时间 7006.13 5805.45 0.76    9.84
线性回归 气象+时间+滞后  815.94  516.61 1.00    1.15
随机森林   仅气象+时间 5313.91 4360.61 0.86    7.46
随机森林 气象+时间+滞后  775.18  492.02 1.00    1.09
梯度提升   仅气象+时间 5935.23 5182.15 0.83    8.33
梯度提升 气象+时间+滞后  737.30  491.49 1.00    1.04


## 5. 结果可视化（图 9–12）

### 图 9　总负荷自相关 ACF

自相关系数在 144 步（1 天）、1008 步（1 周）附近出现明显峰值，说明负荷有稳定的日、周周期，这是选择 1 天 / 1 周滞后特征、以及短步长（10 分钟、1 小时）滞后能提供近邻信息的依据。

In [6]:
lags = list(range(0, 2017, 6))  # 每 1 小时取一个滞后
acf = [1.0] + [df['Z_total'].autocorr(l) for l in lags[1:]]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lags, acf, lw=1.2)
ax.axhline(0, color='k', lw=0.6)
for L, lab in [(144, '144 个点（1 天）'), (1008, '1008 个点（1 周）')]:
    ax.axvline(L, color='#d62728' if L == 144 else '#2ca02c', ls='--', lw=1, label=lab)
ax.set_xlabel('滞后（10 分钟步）')
ax.set_ylabel('自相关系数')
ax.set_title('总负荷自相关 ACF')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig('figures/fig_B1_acf.png', dpi=150)
plt.show()
print('saved figures/fig_B1_acf.png')

saved figures/fig_B1_acf.png


### 图 10　测试集预测 vs 真实（2017-11-20 至 11-22，含滞后）

In [ ]:
wmask = (test.index >= '2017-11-20') & (test.index <= '2017-11-22')
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(test.index[wmask], test['Z_total'][wmask], color='k', lw=1.2, label='真实负荷')
ax.plot(test.index[wmask], pred_store[('线性回归', '气象+时间+滞后')][wmask], color='#1f77b4', lw=1.0, label='线性回归（含滞后）')
ax.plot(test.index[wmask], pred_store[('随机森林', '气象+时间+滞后')][wmask], color='#ff7f0e', lw=1.0, label='随机森林（含滞后）')
ax.set_xlabel('时间')
ax.set_ylabel('总负荷 (kW)')
ax.set_title('测试集预测 vs 真实（2017-11-20 至 11-22）')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig('figures/fig_B2_pred.png', dpi=150)
plt.show()
print('saved figures/fig_B2_pred.png')

### 图 11　预测 vs 真实散点图与残差分布（随机森林·含滞后）

In [ ]:
best = pred_store[('随机森林', '气象+时间+滞后')]
resid = y_test - best
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(y_test, best, s=4, alpha=0.3, color='#1f77b4')
lims = [min(y_test.min(), best.min()), max(y_test.max(), best.max())]
axes[0].plot(lims, lims, 'k--', lw=1)
axes[0].set_xlabel('真实负荷 (kW)')
axes[0].set_ylabel('预测负荷 (kW)')
axes[0].set_title('预测 vs 真实（随机森林·含滞后）')
axes[1].hist(resid, bins=60, color='#ff7f0e', alpha=0.8)
axes[1].axvline(0, color='k', lw=0.8)
axes[1].set_xlabel('残差 (kW)')
axes[1].set_title('残差分布')
for a in axes:
    a.grid(alpha=0.3)
fig.tight_layout()
fig.savefig('figures/fig_B3_scatter_resid.png', dpi=150)
plt.show()
print('saved figures/fig_B3_scatter_resid.png')

### 图 12　各模型 RMSE / MAE 对比

In [ ]:
pivot_rmse = res.pivot(index='模型', columns='特征', values='RMSE')
pivot_mae = res.pivot(index='模型', columns='特征', values='MAE')
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, piv, name in [(axes[0], pivot_rmse, 'RMSE'), (axes[1], pivot_mae, 'MAE')]:
    piv.plot(kind='bar', ax=ax, rot=0)
    ax.set_title(f'各模型 {name} 对比')
    ax.set_xlabel('')
    ax.grid(alpha=0.3, axis='y')
    for c in ax.containers:
        ax.bar_label(c, fmt='%.0f', fontsize=8)
fig.tight_layout()
fig.savefig('figures/fig_B4_metrics.png', dpi=150)
plt.show()
print('saved figures/fig_B4_metrics.png')